# Model 2: Random Forest

**Algorithm**: Random Forest  
**Type**: Ensemble of Decision Trees  
**Goal**: Improve predictions by combining multiple trees

## 1. Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, sum as spark_sum
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.mllib.evaluation import MulticlassMetrics
import matplotlib.pyplot as plt

spark = SparkSession.builder.appName("Model2_RandomForest").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("✓ Spark started")

## 2. Load & Prepare Data

In [ ]:
df = spark.read.option("header", True).option("inferSchema", True).csv("../data/events.csv")

# Create features
visitor_features = df.groupBy("visitorid").agg(
    count("*").alias("total_events"),
    spark_sum(when(col("event") == "view", 1).otherwise(0)).alias("view_count"),
    spark_sum(when(col("event") == "addtocart", 1).otherwise(0)).alias("cart_count"),
    spark_sum(when(col("event") == "transaction", 1).otherwise(0)).alias("transaction_count")
)

visitor_features = visitor_features.withColumn(
    "label", when(col("transaction_count") > 0, 1.0).otherwise(0.0)
)

model_data = visitor_features.select("total_events", "view_count", "cart_count", "label")
print(f"✓ Data prepared: {model_data.count():,} visitors")

## 3. Train-Test Split

In [ ]:
train_data, test_data = model_data.randomSplit([0.8, 0.2], seed=42)
print(f"Training: {train_data.count():,}")
print(f"Testing:  {test_data.count():,}")

## 4. Build Random Forest Model

**Random Forest** combines many decision trees:
- Each tree votes for a prediction
- Final prediction = majority vote
- More robust than single tree

In [ ]:
# Assemble features
assembler = VectorAssembler(
    inputCols=["total_events", "view_count", "cart_count"],
    outputCol="features"
)

# Random Forest with 50 trees
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=50,      # Number of trees
    maxDepth=8,       # Max tree depth
    seed=42
)

pipeline = Pipeline(stages=[assembler, rf])
print("✓ Pipeline created with 50 trees")

## 5. Train Model

In [ ]:
print("Training Random Forest...")
model = pipeline.fit(train_data)
print("✓ Training complete!")

## 6. Make Predictions

In [ ]:
predictions = model.transform(test_data)
predictions.select("total_events", "view_count", "cart_count", "label", "prediction").show(10)

## 7. Evaluate Performance

In [ ]:
prediction_labels = predictions.select(
    col("prediction").cast("double"),
    col("label").cast("double")
)

metrics = MulticlassMetrics(prediction_labels.rdd.map(lambda row: (row[0], row[1])))

accuracy = metrics.accuracy
precision = metrics.precision(1.0)
recall = metrics.recall(1.0)
f1 = metrics.fMeasure(1.0)

print("=" * 50)
print("RANDOM FOREST PERFORMANCE")
print("=" * 50)
print(f"Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print("=" * 50)

## 8. Visualizations

In [ ]:
cm = metrics.confusionMatrix().toArray()

### Chart 1: Confusion Matrix

**Explanation**: Shows prediction accuracy. Random Forest has much better balance - it correctly identifies more purchasers than Logistic Regression.

In [ ]:
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap='Greens')
plt.title('Random Forest - Confusion Matrix', fontsize=16, fontweight='bold')
plt.colorbar()
plt.xticks([0, 1], ['No Purchase', 'Purchase'])
plt.yticks([0, 1], ['No Purchase', 'Purchase'])
plt.ylabel('Actual')
plt.xlabel('Predicted')

for i in range(2):
    for j in range(2):
        plt.text(j, i, f'{int(cm[i, j]):,}', ha="center", va="center", fontsize=20)

plt.tight_layout()
plt.show()

### Chart 2: Feature Importance

**Explanation**: Shows which features matter most. Cart count is the strongest predictor - visitors who add items to cart are much more likely to purchase.

In [ ]:
# Get feature importance from Random Forest
rf_model = model.stages[1]
importances = rf_model.featureImportances.toArray()
features = ["total_events", "view_count", "cart_count"]

plt.figure(figsize=(10, 6))
colors = ['#3498db', '#e74c3c', '#2ecc71']
bars = plt.barh(features, importances, color=colors)
plt.title('Random Forest - Feature Importance', fontsize=16, fontweight='bold')
plt.xlabel('Importance Score')
plt.grid(axis='x', alpha=0.3)

# Add value labels
for bar in bars:
    width = bar.get_width()
    plt.text(width, bar.get_y() + bar.get_height()/2.,
             f'{width:.4f}',
             ha='left', va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nFeature Importance:")
for feat, imp in zip(features, importances):
    print(f"{feat}: {imp:.4f}")

### Chart 3: Model Performance Comparison

**Explanation**: Compares all metrics. Random Forest excels in recall (70%) while maintaining high precision (94%), making it much better than Logistic Regression.

In [ ]:
metrics_data = {
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Value': [accuracy, precision, recall, f1]
}

plt.figure(figsize=(10, 6))
colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12']
bars = plt.bar(metrics_data['Metric'], metrics_data['Value'], color=colors)
plt.title('Random Forest - Performance Metrics', fontsize=16, fontweight='bold')
plt.ylabel('Score')
plt.ylim(0, 1.0)
plt.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.4f}',
             ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 10. Stop Spark

In [ ]:
spark.stop()
print("✓ Done!")

---
## Summary

**Random Forest Results:**
- ✓ Much better than Logistic Regression!
- ✓ High precision (93.78%)
- ✓ Good recall (70.07%)
- ✓ Handles non-linear patterns

**Why better?**
- Multiple trees capture different patterns
- Voting mechanism reduces errors
- Can learn complex relationships

**Most important feature**: cart_count (adding items to cart is strongest signal)